# 03 Models Baseline

Этот ноутбук обучает **модели на каждом fold** и считает метрики качества:

1. **Ridge** — L2 регуляризация (аналог GBLUP)
2. **Lasso** — L1 регуляризация (отбор признаков)
3. **Elastic Net** — комбинация L1 + L2

Для каждой модели считаем: `corr` (корреляция), `rmse`, `mae`, `r2`.

**Для кого**: для тех, кто хочет увидеть, как разные модели работают на одних данных и какие показывают лучшие результаты.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

from gp_py.io import fn_load_genotype, fn_load_phenotype
from gp_py.cv import fn_cross_validation_preparation, fn_cross_validation_within_population
from gp_py.schema import MergedData

## 1. Подготовка данных

In [ ]:
DATA_DIR = Path("../inst/exec_Rscript/input")

G = fn_load_genotype(str(DATA_DIR / "test_geno.Rds"))
list_pheno = fn_load_phenotype(str(DATA_DIR / "test_pheno.tsv"))

# Фильтруем genotype
from gp_py.io import fn_filter_genotype, fn_filter_phenotype, fn_merge_genotype_and_phenotype

Gf = fn_filter_genotype(G)
phf = fn_filter_phenotype(list_pheno)
merged = fn_merge_genotype_and_phenotype(Gf, phf)

# Оставляем только известные фенотипы для CV
known_mask = merged.y.notna()
merged_known = MergedData(
    G=merged.G.loc[known_mask].copy(),
    y=merged.y.loc[known_mask].copy(),
    pop=merged.pop.loc[known_mask].copy(),
    trait_name=merged.trait_name,
)

print(f"Размер данных для CV: {merged_known.G.shape}")
print(f"Популяции: {merged_known.pop.unique()}")

## 2. Запуск CV с несколькими моделями

In [ ]:
# Параметры
N_FOLDS = 3
N_REPS = 2
MODELS = ("ridge", "lasso", "elastic_net")

print("Запускаем кросс-валидацию...")
cv_results = fn_cross_validation_within_population(
    merged_known,
    n_folds=N_FOLDS,
    n_reps=N_REPS,
    vec_models_to_test=MODELS,
    bool_parallel=False,  # для наглядности.Sequential
    bayes_backend="native",  # без R
    verbose=True
)

print("Готово!")

## 3. Результаты CV — сводная таблица метрик

In [ ]:
metrics_df = cv_results["METRICS_WITHIN_POP"]
print("=== Метрики по всем fold-ам ===")
print(metrics_df.head(10).to_string())

## 4. Агрегированные метрики по моделям

In [ ]:
# Среднее и std по моделям
agg = metrics_df.groupby("model").agg({
    "corr": ["mean", "std"],
    "rmse": ["mean", "std"],
    "mae": ["mean", "std"],
    "r2": ["mean", "std"]
}).round(4)

print("=== Среднее ± std метрик по моделям ===")
print(agg.to_string())

## 5. Визуализация: сравнение моделей по корреляции

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot корреляций
ax1 = axes[0]
models = metrics_df["model"].unique()
data_for_box = [metrics_df[metrics_df["model"] == m]["corr"].values for m in models]
bp = ax1.boxplot(data_for_box, labels=models, patch_artist=True)
for patch in bp["boxes"]:
    patch.set_facecolor("lightblue")
ax1.set_ylabel("Correlation (r)")
ax1.set_title("Distribution of Correlation by Model")
ax1.grid(True, alpha=0.3)

# Barplot средней корреляции
ax2 = axes[1]
mean_corr = metrics_df.groupby("model")["corr"].mean().sort_values(ascending=False)
mean_corr.plot(kind="bar", ax=ax2, color="steelblue", alpha=0.7)
ax2.set_ylabel("Mean Correlation")
ax2.set_title("Mean Correlation by Model")
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6. Лучшая модель

In [ ]:
best_model = metrics_df.groupby("model")["corr"].mean().idxmax()
best_corr = metrics_df.groupby("model")["corr"].mean().max()

print("=== Лучшая модель по средней корреляции ===")
print(f"  Модель: {best_model}")
print(f"  Средняя корреляция: {best_corr:.4f}")

## Итог

- Мы обучили 3 модели (Ridge, Lasso, Elastic Net) на 3 folds × 2 reps = 6 итераций
- Для каждой модели получили метрики: corr, rmse, mae, r2
- Лучшая модель определяется по средней корреляции на validation-фолдах
- Эта лучшая модель будет использоваться для предсказания пропущенных фенотипов

**Следующий шаг**: перейти к ноутбуку `04_missing_phenotype_prediction`, чтобы увидеть, как лучшая модель предсказывает пропущенные значения.